# SMART Constrained Variant Training (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/wosac-sim-agents-experiments/blob/main/experiments/smart-constrained/notebooks/smart-constrained_colab.ipynb)

## Objective
- Train constrained SMART variants and persist checkpoints to Drive.
- Keep this notebook training-only.
- Run simulation/evaluation/comparison in dedicated notebooks.


In [ ]:
# Step 1: Sync this repo and bootstrap deterministic Colab runtime
import os
import subprocess
import sys

REPO_URL = "https://github.com/achyutmorang/wosac-sim-agents-experiments.git"
REPO_DIR = "/content/wosac-sim-agents-experiments"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for p in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.platform import bootstrap_colab_runtime_with_config, wosac_colab_runtime_config
runtime_cfg = wosac_colab_runtime_config(repo_url=REPO_URL, repo_branch="main", repo_dir=REPO_DIR)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
print("repo_rev:", bootstrap.repo_sync.repo_rev)

if bootstrap.setup.result.get("restart_required", False):
    raise RuntimeError("Runtime restart required. Restart runtime and rerun this cell.")


In [ ]:
# Step 2: Load config, select run mode/sweep profile, stage data from GCS, and build preprocess policy
import hashlib
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

from src.workflows import bootstrap_experiment_pack, load_experiment_config


def _bool_env(name: str, default: bool = False) -> bool:
    text = os.environ.get(name, "").strip().lower()
    if text in {"1", "true", "yes", "y", "on"}:
        return True
    if text in {"0", "false", "no", "n", "off"}:
        return False
    return bool(default)


def _ensure_colab_auth() -> None:
    if 'google.colab' not in sys.modules:
        return
    try:
        from google.colab import auth  # type: ignore

        auth.authenticate_user()
        print('colab_gcs_auth: OK')
    except Exception as exc:
        print(f'colab_gcs_auth: skipped ({exc})')


def _gcloud_ls(pattern: str) -> list[str]:
    try:
        out = subprocess.check_output(
            ["bash", "-lc", f"gcloud storage ls '{pattern}'"],
            text=True,
            stderr=subprocess.STDOUT,
        )
        rows = [line.strip() for line in out.splitlines() if line.strip()]
        return rows
    except Exception:
        return []


def _choose_shards(pattern: str, limit: int) -> list[str]:
    rows = sorted(_gcloud_ls(pattern))
    if limit > 0:
        return rows[:limit]
    return rows


def _local_shards(split_dir: Path) -> list[Path]:
    return sorted([p for p in split_dir.glob('*.tfrecord*') if p.is_file()])


def _count_processed(split_dir: Path) -> int:
    files = [p for p in split_dir.glob('*.pkl') if p.is_file()]
    files += [p for p in split_dir.glob('*.pickle') if p.is_file()]
    return len(files)


def _parse_float_list_env(name: str, default: list[float]) -> list[float]:
    text = os.environ.get(name, '').strip()
    if not text:
        return list(default)
    out = []
    for part in text.split(','):
        part = part.strip()
        if not part:
            continue
        try:
            out.append(float(part))
        except Exception:
            pass
    return out or list(default)


def _parse_int_list_env(name: str, default: list[int]) -> list[int]:
    text = os.environ.get(name, '').strip()
    if not text:
        return list(default)
    out = []
    for part in text.split(','):
        part = part.strip()
        if not part:
            continue
        try:
            out.append(int(float(part)))
        except Exception:
            pass
    return out or list(default)


def _stage_split(*, gcs_root: str, split_name: str, shard_limit: int, local_split_dir: Path, force: bool) -> dict:
    local_split_dir.mkdir(parents=True, exist_ok=True)
    pattern = f"{gcs_root}/{split_name}/{split_name}.tfrecord-*"
    remote_shards = _choose_shards(pattern, shard_limit)
    existing = {p.name for p in _local_shards(local_split_dir)}
    pending = remote_shards if force else [u for u in remote_shards if Path(u).name not in existing]

    copied = 0
    errors: list[str] = []
    for uri in pending:
        try:
            subprocess.run(
                ["gcloud", "storage", "cp", "--no-clobber", uri, str(local_split_dir) + "/"],
                check=True,
            )
            copied += 1
        except Exception as exc:
            errors.append(f"{uri}: {type(exc).__name__}")

    local_files = _local_shards(local_split_dir)
    return {
        "split": split_name,
        "pattern": pattern,
        "requested_limit": int(shard_limit),
        "remote_candidates": len(remote_shards),
        "pending_before_copy": len(pending),
        "copied": int(copied),
        "local_count": len(local_files),
        "local_sample": [str(p) for p in local_files[:5]],
        "errors": errors,
    }


EXPERIMENT_SLUG = "smart-constrained"
bundle = bootstrap_experiment_pack(slug=EXPERIMENT_SLUG, repo_root=".")
cfg = load_experiment_config(slug=EXPERIMENT_SLUG, repo_root=".")
display(bundle.summary_table)

RUN = dict(cfg.get("run", {}))
SMART = dict(cfg.get("smart", {}))
CONSTRAINTS = dict(cfg.get("constraints", {}))
SWEEP = dict(cfg.get("sweep", {}))

RUN_NAME = str(RUN.get("run_name", "dev"))
RUN_PREFIX = str(RUN.get("run_prefix", "smart_constrained"))
PERSIST_ROOT = str(RUN.get("persist_root", "/content/drive/MyDrive/wosac_experiments"))
RESUME_FROM_EXISTING = bool(RUN.get("resume_from_existing", True))
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
cfg_hash = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode("utf-8")).hexdigest()
persist_run_dir = Path(PERSIST_ROOT) / f"{RUN_PREFIX}_{RUN_NAME}"
persist_run_dir.mkdir(parents=True, exist_ok=True)
RUN_OUTPUT_DIR = persist_run_dir / "outputs" / RUN_TAG
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VARIANT_CKPT_ROOT = os.environ.get("SMART_VARIANT_CKPT_ROOT", "").strip() or str(persist_run_dir / "checkpoints" / "variants")

RUN_MODE = os.environ.get("WOSAC_RUN_MODE", "pilot").strip().lower() or "pilot"
if RUN_MODE not in {"pilot", "full"}:
    print(f"Unknown WOSAC_RUN_MODE={RUN_MODE!r}; falling back to 'pilot'.")
    RUN_MODE = "pilot"

if RUN_MODE == "full":
    sweep_defaults = {
        "temperatures": [0.8, 1.0, 1.2],
        "top_ks": [8, 16],
        "constraint_weights": [0.05, 0.1, 0.2],
    }
else:
    sweep_defaults = {
        "temperatures": [float(v) for v in SWEEP.get("temperatures", [0.9, 1.0])],
        "top_ks": [int(v) for v in SWEEP.get("top_ks", [8])],
        "constraint_weights": [float(v) for v in SWEEP.get("constraint_weights", [0.1, 0.2])],
    }

SWEEP_EFFECTIVE = {
    "temperatures": _parse_float_list_env("SMART_CONSTRAINED_TEMPERATURES", sweep_defaults["temperatures"]),
    "top_ks": _parse_int_list_env("SMART_CONSTRAINED_TOP_KS", sweep_defaults["top_ks"]),
    "constraint_weights": _parse_float_list_env("SMART_CONSTRAINED_WEIGHTS", sweep_defaults["constraint_weights"]),
}

raw_root_override = os.environ.get("SMART_RAW_DATA_ROOT", "").strip()
processed_root_override = os.environ.get("SMART_PROCESSED_DATA_ROOT", "").strip()
if raw_root_override:
    SMART["raw_data_root"] = raw_root_override
if processed_root_override:
    SMART["processed_data_root"] = processed_root_override

DATA_SOURCE_MODE = os.environ.get("SMART_DATA_SOURCE", "gcs_stage").strip().lower() or "gcs_stage"
RUN_DATA_STAGE = _bool_env("SMART_RUN_DATA_STAGE", True)
FORCE_DATA_REDOWNLOAD = _bool_env("SMART_FORCE_DATA_REDOWNLOAD", False)
GCS_DATASET_ROOT = os.environ.get("SMART_GCS_DATASET_ROOT", "gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario").strip().rstrip("/")
TRAIN_SPLIT = os.environ.get("SMART_GCS_TRAIN_SPLIT", "training").strip() or "training"
VAL_SPLIT = os.environ.get("SMART_GCS_VAL_SPLIT", "validation").strip() or "validation"
TRAIN_SHARD_LIMIT = int(os.environ.get("SMART_GCS_TRAIN_SHARDS", "8" if RUN_MODE == "pilot" else "64"))
VAL_SHARD_LIMIT = int(os.environ.get("SMART_GCS_VAL_SHARDS", "2" if RUN_MODE == "pilot" else "8"))

RAW_ROOT = Path(str(SMART.get("raw_data_root", "/content/SMART/data/waymo/scenario"))).expanduser()
PROCESSED_ROOT = Path(str(SMART.get("processed_data_root", "/content/SMART/data/waymo_processed"))).expanduser()
RAW_TRAIN_DIR = RAW_ROOT / "training"
RAW_VAL_DIR = RAW_ROOT / "validation"
PROCESSED_TRAIN_DIR = PROCESSED_ROOT / "training"
PROCESSED_VAL_DIR = PROCESSED_ROOT / "validation"

for d in [RAW_TRAIN_DIR, RAW_VAL_DIR, PROCESSED_TRAIN_DIR, PROCESSED_VAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

data_stage_manifest = {
    "data_source_mode": DATA_SOURCE_MODE,
    "run_data_stage": bool(RUN_DATA_STAGE),
    "force_data_redownload": bool(FORCE_DATA_REDOWNLOAD),
    "gcs_dataset_root": GCS_DATASET_ROOT,
    "train_split": TRAIN_SPLIT,
    "val_split": VAL_SPLIT,
    "train_shard_limit": int(TRAIN_SHARD_LIMIT),
    "val_shard_limit": int(VAL_SHARD_LIMIT),
    "results": {},
}

if DATA_SOURCE_MODE == "gcs_stage" and RUN_DATA_STAGE:
    _ensure_colab_auth()
    data_stage_manifest["results"]["training"] = _stage_split(
        gcs_root=GCS_DATASET_ROOT,
        split_name=TRAIN_SPLIT,
        shard_limit=TRAIN_SHARD_LIMIT,
        local_split_dir=RAW_TRAIN_DIR,
        force=FORCE_DATA_REDOWNLOAD,
    )
    data_stage_manifest["results"]["validation"] = _stage_split(
        gcs_root=GCS_DATASET_ROOT,
        split_name=VAL_SPLIT,
        shard_limit=VAL_SHARD_LIMIT,
        local_split_dir=RAW_VAL_DIR,
        force=FORCE_DATA_REDOWNLOAD,
    )
else:
    data_stage_manifest["results"]["note"] = "staging_skipped"

raw_train_count = len(_local_shards(RAW_TRAIN_DIR))
raw_val_count = len(_local_shards(RAW_VAL_DIR))
processed_train_count = _count_processed(PROCESSED_TRAIN_DIR)
processed_val_count = _count_processed(PROCESSED_VAL_DIR)

SMART_FORCE_PREPROCESS = _bool_env("SMART_FORCE_PREPROCESS", False)
DEFAULT_RUN_PREPROCESS = bool(SMART_FORCE_PREPROCESS or processed_train_count == 0 or processed_val_count == 0)

preprocess_policy = {
    "force_preprocess": bool(SMART_FORCE_PREPROCESS),
    "default_run_preprocess": bool(DEFAULT_RUN_PREPROCESS),
    "processed_train_count": int(processed_train_count),
    "processed_val_count": int(processed_val_count),
    "raw_train_count": int(raw_train_count),
    "raw_val_count": int(raw_val_count),
}

print("RUN_TAG:", RUN_TAG)
print("persist_run_dir:", persist_run_dir)
print("RUN_OUTPUT_DIR:", RUN_OUTPUT_DIR)
print("RUN_MODE:", RUN_MODE)
print("VARIANT_CKPT_ROOT:", VARIANT_CKPT_ROOT)
print("RESUME_FROM_EXISTING:", RESUME_FROM_EXISTING)
print("SWEEP_EFFECTIVE:", json.dumps(SWEEP_EFFECTIVE, sort_keys=True))
print("SMART repo:", SMART.get("repo_url", ""), SMART.get("branch", ""))
print("SMART train config:", SMART.get("train_config", ""))
print("SMART val config:", SMART.get("val_config", ""))
print("SMART raw_data_root:", SMART.get("raw_data_root", ""))
print("SMART processed_data_root:", SMART.get("processed_data_root", ""))
print("cfg_hash:", cfg_hash)
print("data_stage_manifest:")
print(json.dumps(data_stage_manifest, indent=2, sort_keys=True))
print("preprocess_policy:")
print(json.dumps(preprocess_policy, indent=2, sort_keys=True))



In [ ]:
# Step 3: Build constrained variant training grid (train-only)
from src.workflows import run_smart_constrained_flow

flow_bundle = run_smart_constrained_flow(
    run_tag=RUN_TAG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root=".",
    resume_from_existing=RESUME_FROM_EXISTING,
    variant_metrics_dir="",
    smart_checkpoint_root=VARIANT_CKPT_ROOT,
    max_collision_rate=CONSTRAINTS.get("max_collision_rate"),
    max_offroad_rate=CONSTRAINTS.get("max_offroad_rate"),
    max_tl_violation_rate=CONSTRAINTS.get("max_tl_violation_rate"),
    min_diversity_score=CONSTRAINTS.get("min_diversity_score"),
    temperatures=SWEEP_EFFECTIVE.get("temperatures", [0.9, 1.0]),
    top_ks=SWEEP_EFFECTIVE.get("top_ks", [8]),
    constraint_weights=SWEEP_EFFECTIVE.get("constraint_weights", [0.1, 0.2]),
    smart_repo_url=str(SMART.get("repo_url", "https://github.com/rainmaker22/SMART.git")),
    smart_repo_branch=str(SMART.get("branch", "main")),
    smart_repo_commit=str(SMART.get("repo_commit", "")).strip(),
    smart_repo_dir=str(SMART.get("repo_dir", "/content/SMART")),
    smart_train_config=str(SMART.get("train_config", "configs/train/train_scalable.yaml")),
    smart_val_config=str(SMART.get("val_config", "configs/validation/validation_scalable.yaml")),
    smart_ckpt_path="",
    smart_raw_data_root=str(SMART.get("raw_data_root", "/content/SMART/data/waymo/scenario")),
    smart_processed_data_root=str(SMART.get("processed_data_root", "/content/SMART/data/waymo_processed")),
    smart_install_pyg=bool(SMART.get("install_pyg", True)),
    smart_env_lockfile=str(SMART.get("env_lockfile", "")).strip(),
    smart_train_seed=int(SMART.get("seed", 2)),
    smart_deterministic_train=bool(SMART.get("deterministic_train", True)),
    smart_train_launcher_path=str(SMART.get("train_launcher_path", "scripts/smart_train_repro.py")),
    smart_force_preprocess=bool(preprocess_policy.get("force_preprocess", False)),
    sync_smart_repo=True,
)

print("summary:")
print(json.dumps(flow_bundle.summary, indent=2, sort_keys=True))

if flow_bundle.summary.get("status") == "sync_failed":
    raise RuntimeError(
        f"SMART repo sync failed before command execution: {flow_bundle.summary.get('sync_error', 'unknown')}"
    )
print("num_variants:", len(flow_bundle.variants))
print("first_variant:")
print(json.dumps(flow_bundle.variants[0], indent=2, sort_keys=True))


In [ ]:
# Step 4: Optional constrained training execution (no evaluation in this notebook)
import os
import json
import subprocess

RUN_SETUP = _bool_env("SMART_RUN_SETUP", False)
RUN_PREPROCESS = _bool_env("SMART_RUN_PREPROCESS", bool(preprocess_policy.get("default_run_preprocess", False)))
RUN_TRAIN = _bool_env("SMART_RUN_TRAIN", False)
AUTO_SETUP = _bool_env("SMART_AUTO_SETUP", True)

if bool(preprocess_policy.get("force_preprocess", False)):
    RUN_PREPROCESS = True

if RUN_TRAIN and not RUN_SETUP and AUTO_SETUP:
    RUN_SETUP = True
    print("SMART_AUTO_SETUP=1 -> enabling setup before train.")

if RUN_PREPROCESS:
    if int(preprocess_policy.get("raw_train_count", 0)) <= 0 or int(preprocess_policy.get("raw_val_count", 0)) <= 0:
        raise RuntimeError(
            "RUN_PREPROCESS requested but raw shards are missing. "
            "Rerun Step 2 with SMART_RUN_DATA_STAGE=1 or point SMART_RAW_DATA_ROOT to existing local shards."
        )
    for _d in [PROCESSED_TRAIN_DIR, PROCESSED_VAL_DIR]:
        _d.mkdir(parents=True, exist_ok=True)

variant_ids_env = [v.strip() for v in os.environ.get("SMART_VARIANT_IDS_TO_RUN", "").split(",") if v.strip()]
VARIANT_IDS_TO_RUN = list(variant_ids_env)

stage_flags = {
    "run_setup": bool(RUN_SETUP),
    "run_preprocess": bool(RUN_PREPROCESS),
    "run_train": bool(RUN_TRAIN),
    "auto_setup": bool(AUTO_SETUP),
    "variant_ids_to_run": list(VARIANT_IDS_TO_RUN),
    "run_mode": RUN_MODE,
}
print("stage_flags:")
print(json.dumps(stage_flags, indent=2, sort_keys=True))


def _run_cmd(cmd: str, idx: int, total: int) -> None:
    print(f"[exec {idx}/{total}] {cmd}")
    merged_cmd = f"export PYTHONUNBUFFERED=1; {cmd}"
    process = subprocess.Popen(
        ["bash", "-lc", merged_cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    recent_lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        recent_lines.append(line)
        if len(recent_lines) > 80:
            recent_lines.pop(0)
    return_code = int(process.wait())
    if return_code == 0:
        return
    smart_repo_dir = str(SMART.get("repo_dir", "/content/SMART"))
    diag = {
        "failed_command": cmd,
        "exit_code": return_code,
        "smart_repo_dir": smart_repo_dir,
        "smart_repo_exists": bool(os.path.isdir(smart_repo_dir)),
        "data_preprocess_exists": bool(os.path.isfile(os.path.join(smart_repo_dir, "data_preprocess.py"))),
        "raw_train_dir": str(RAW_TRAIN_DIR),
        "raw_val_dir": str(RAW_VAL_DIR),
        "raw_train_count": int(preprocess_policy.get("raw_train_count", 0)),
        "raw_val_count": int(preprocess_policy.get("raw_val_count", 0)),
        "processed_train_dir": str(PROCESSED_TRAIN_DIR),
        "processed_val_dir": str(PROCESSED_VAL_DIR),
        "processed_train_count": int(preprocess_policy.get("processed_train_count", 0)),
        "processed_val_count": int(preprocess_policy.get("processed_val_count", 0)),
        "recent_output": "".join(recent_lines[-20:]),
    }
    print("step4_diagnostics:")
    print(json.dumps(diag, indent=2, sort_keys=True))
    raise subprocess.CalledProcessError(return_code, ["bash", "-lc", merged_cmd])


variant_map = {v["variant_id"]: v for v in flow_bundle.variants}
if VARIANT_IDS_TO_RUN:
    selected_variants = [variant_map[v] for v in VARIANT_IDS_TO_RUN if v in variant_map]
else:
    selected_variants = [flow_bundle.variants[0]] if flow_bundle.variants else []

for variant in selected_variants:
    print("Running variant:", variant["variant_id"])
    cmds = []
    if RUN_SETUP:
        cmds.append(variant["setup_cmd"])
    if RUN_PREPROCESS:
        cmds.extend([variant["preprocess_train_cmd"], variant["preprocess_val_cmd"]])
    if RUN_TRAIN:
        cmds.append(variant["train_cmd"])

    for i, cmd in enumerate(cmds, start=1):
        _run_cmd(cmd, i, len(cmds))

print("Constrained training execution block done.")


In [ ]:
# Step 5: Write constrained training artifact + run manifest (Drive)
from pathlib import Path

from src.workflows import build_training_run_manifest, write_json

stage_flags = globals().get("stage_flags", {"run_setup": False, "run_preprocess": False, "run_train": False, "variant_ids_to_run": [], "run_mode": RUN_MODE})
variant_ckpt_scan = {}
for v in flow_bundle.variants:
    ckpt_dir = Path(v.get("checkpoint_dir", "")).expanduser()
    ckpts = sorted([str(p) for p in ckpt_dir.glob("*.ckpt")]) if str(ckpt_dir) else []
    variant_ckpt_scan[v["variant_id"]] = {
        "checkpoint_dir": str(ckpt_dir),
        "checkpoint_files": ckpts,
        "resume_checkpoint_path": str(v.get("resume_checkpoint_path", "")),
        "resume_source": str(v.get("resume_source", "")),
    }

payload = {
    "run_id": "smart_constrained_train_v0",
    "status": str(flow_bundle.summary.get("status", "unknown")),
    "run_tag": RUN_TAG,
    "run_mode": RUN_MODE,
    "sweep_effective": SWEEP_EFFECTIVE,
    "checkpoint_root": VARIANT_CKPT_ROOT,
    "run_output_dir": str(RUN_OUTPUT_DIR),
    "stage_flags": stage_flags,
    "data_stage_manifest": data_stage_manifest,
    "preprocess_policy": preprocess_policy,
    "variants": flow_bundle.variants,
    "variant_ckpt_scan": variant_ckpt_scan,
    "flow_summary": flow_bundle.summary,
    "flow_artifacts": flow_bundle.artifacts,
    "notes": [
        "Training-only notebook. Use smart_rollout_simulation_colab.ipynb, smart_checkpoint_eval_colab.ipynb, and smart_comparative_eval_colab.ipynb for simulation/eval/comparison.",
    ],
    "provenance": {
        "config_hash": cfg_hash,
        "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "experiment_slug": EXPERIMENT_SLUG,
    },
}

drive_path = RUN_OUTPUT_DIR / "smart_constrained_train_v0.json"
drive_path.parent.mkdir(parents=True, exist_ok=True)
drive_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

baseline_summary = dict((flow_bundle.baseline or {}).get("summary", {}) or {})
run_manifest = build_training_run_manifest(
    run_id="smart_constrained_train_v0",
    run_tag=RUN_TAG,
    experiment_slug=EXPERIMENT_SLUG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root=".",
    config_hash=cfg_hash,
    flow_summary=baseline_summary,
    stage_flags=stage_flags,
    checkpoint_dir=VARIANT_CKPT_ROOT,
    resume_checkpoint_path="",
    resume_checkpoint_source="variant_specific",
    extra={
        "flow_artifacts": flow_bundle.artifacts,
        "num_variants": len(flow_bundle.variants),
        "variant_resume_summary": {k: {"resume_checkpoint_path": v.get("resume_checkpoint_path", ""), "resume_source": v.get("resume_source", "")} for k, v in variant_ckpt_scan.items()},
        "sweep_effective": SWEEP_EFFECTIVE,
        "data_stage_manifest": data_stage_manifest,
        "preprocess_policy": preprocess_policy,
    },
)
manifest_path = RUN_OUTPUT_DIR / "run_manifest.json"
write_json(manifest_path, run_manifest)

print("drive_path:", drive_path)
print("manifest_path:", manifest_path)
print("variants_scanned:", len(variant_ckpt_scan))
